### 1. SETUP PATHS
This chunk executes the relevant logic for the step mentioned above.

In [1]:
import pandas as pd
import numpy as np
import os

# 1. SETUP PATHS
current_dir = os.getcwd()
data_dir = os.path.join(current_dir, "..", "csv")

input_file = os.path.join(data_dir, "step3_claims_meds_and_comps.xlsx")
output_file = os.path.join(data_dir, "step4_patient_aggregated.xlsx")
key_file = os.path.join(data_dir, "patient_id_key.xlsx")

print("--- STEP 4: FINAL AGGREGATION & METRICS ---")

if os.path.exists(input_file):
    df = pd.read_excel(input_file)
    print(f" Loaded {len(df)} visits.")
else:
    print(" Error: Step 3 file missing.")

--- STEP 4: FINAL AGGREGATION & METRICS ---
✅ Loaded 315402 visits.


### Chunk 2: ⚙ Aggregating to Patient Level
This chunk executes the relevant logic for the step mentioned above.

In [2]:
print("⚙ Aggregating to Patient Level...")

# 1. Calculate Unique Diagnoses (Before dropping columns)
# We calculate this on the full dataset before squashing it
diag_cols = [c for c in df.columns if c.startswith("DIAG_")]
diag_counts = df.groupby("PATIENT_CODE")[diag_cols].apply(
    lambda x: len(set(x.values.flatten()) - {np.nan})
)

# 2. Define Aggregation Rules
# Helper for Admissions
df["Is_IP"] = df["CLAIM_TYPE_CODE"].astype(str).str.upper().apply(lambda x: 1 if "IP" in x else 0)

agg_rules = {
    "SEX": "first",
    "AGE": "max",         # Age at most recent visit
    "LOS": "mean",        # Average Length of Stay
    "Is_IP": "sum",       # Total Admissions
    "CLAIM_CODE": "count" # Total Visits
}

# Add Max rule for all Flags (Meds & Complications)
for col in df.columns:
    if col.startswith("MED_") or col.startswith("COMP_"):
        agg_rules[col] = "max"

# 3. Execute Aggregation
df_patient = df.groupby("PATIENT_CODE").agg(agg_rules).reset_index()

# 4. Rename Columns
df_patient.rename(columns={
    "LOS": "Avg_LOS",
    "Is_IP": "Num_Admissions",
    "CLAIM_CODE": "Num_Visits"
}, inplace=True)

# 5. Attach Unique Diagnosis Count
df_patient["Total_Unique_Diagnoses"] = df_patient["PATIENT_CODE"].map(diag_counts)

print(f" Aggregated into {len(df_patient)} unique patients.")

⚙️ Aggregating to Patient Level...
✅ Aggregated into 62135 unique patients.


### Chunk 3: Calculating Derived Features
This chunk executes the relevant logic for the step mentioned above.

In [3]:
print(" Calculating Derived Features...")

# 1. Round Avg_LOS to 1 decimal place (e.g., 1.1)
df_patient["Avg_LOS"] = df_patient["Avg_LOS"].round(1)

# 2. Basic Flags
df_patient["Readmitted_Yes_No"] = (df_patient["Num_Admissions"] > 1).astype(int)
df_patient["Admitted_Yes_No"] = (df_patient["Num_Admissions"] > 0).astype(int)

# 3. Total Meds Count (Sum of all MED_ columns)
med_cols = [c for c in df_patient.columns if c.startswith("MED_")]
df_patient["Total_Meds_Count"] = df_patient[med_cols].sum(axis=1)

# 4. Complication Flag (Any complication > 0)
comp_cols = [c for c in df_patient.columns if c.startswith("COMP_")]
df_patient["Complication_Yes_No"] = (df_patient[comp_cols].sum(axis=1) > 0).astype(int)

# --- 5. SEVERITY INDEX LOGIC ---
print("   -> Calculating Severity Index...")

# Step A: Default everyone to "Moderate" (Middle ground)
df_patient["SEVERITY_INDEX"] = "Moderate"

# Step B: Handle "Mild"
# Rule: Takes ONLY Biguanide (Total=1 AND Biguanide=1)
# Also treating 0 Meds as "Mild" (Diet Controlled)
df_patient.loc[df_patient["Total_Meds_Count"] == 0, "SEVERITY_INDEX"] = "Mild"

if "MED_BIGUANIDE" in df_patient.columns:
    df_patient.loc[
        (df_patient["MED_BIGUANIDE"] == 1) & 
        (df_patient["Total_Meds_Count"] == 1), 
        "SEVERITY_INDEX"
    ] = "Mild"

# Step C: Handle "Severe" (The Override)
# Rule: If taking Insulin, they are Severe (regardless of other drugs)
if "MED_INSULIN_THERAPY" in df_patient.columns:
    df_patient.loc[df_patient["MED_INSULIN_THERAPY"] == 1, "SEVERITY_INDEX"] = "Severe"

display(df_patient["SEVERITY_INDEX"].value_counts())

✨ Calculating Derived Features...
   -> Calculating Severity Index...


SEVERITY_INDEX
Mild        43711
Moderate    17758
Severe        666
Name: count, dtype: int64

### Chunk 4: Finalizing & Reordering
This chunk executes the relevant logic for the step mentioned above.

In [4]:
print(" Finalizing & Reordering...")

# 1. Generate Patient_ID (P00001...)
df_patient["Patient_ID"] = ["P" + str(i).zfill(5) for i in range(1, len(df_patient) + 1)]

# 2. SAVE THE KEY (Crucial for reversing anonymization later)
df_key = df_patient[["Patient_ID", "PATIENT_CODE"]]
df_key.to_excel(key_file, index=False)
print(f" Anonymization Key saved to: {key_file}")

# 3. DEFINE EXACT COLUMN ORDER
desired_order = [
    "Patient_ID", "SEX", "AGE", "Avg_LOS", "Num_Admissions", "Num_Visits", 
    "Readmitted_Yes_No", "Admitted_Yes_No", 
    "MED_ALPHA_GLUCOSIDE_INHIBITORS", "MED_BIGUANIDE", "MED_COMBINATION_DRUG", 
    "MED_DPP_4_INHIBITORS", "MED_GLP_1_RECEPTOR_AGONISTS", "MED_INSULIN_THERAPY", 
    "MED_MEGLITINIDES", "MED_SGLT_2_INHIBITORS", "MED_SULPHONYLUREAS", "MED_THIAZOLIDINEDIONES", 
    "Total_Meds_Count", 
    "COMP_DIABETIC_FOOT", "COMP_MACROVASCULAR", "COMP_NEPHROPATHY", 
    "COMP_NEUROPATHY", "COMP_RETINOPATHY", 
    "Total_Unique_Diagnoses", "Complication_Yes_No", "SEVERITY_INDEX"
]

# 4. Select & Reorder
# We use a list comprehension to only select columns that actually exist
# (This prevents errors if one specific medication column is missing)
final_cols = [c for c in desired_order if c in df_patient.columns]

# Check for missing columns
missing = set(desired_order) - set(df_patient.columns)
if missing:
    print(f" Note: The following columns were not found in the data and will be skipped: {missing}")

df_final = df_patient[final_cols]

# 5. Save Final File
print(f"💾 Saving to: {output_file}")
df_final.to_excel(output_file, index=False)
print(" Done! Dataset is ready.")

🧹 Finalizing & Reordering...
🔐 Anonymization Key saved to: c:\Users\thiranbarath\Documents\GitHub\dataPreprocessing\source code\..\csv\patient_id_key.xlsx
💾 Saving to: c:\Users\thiranbarath\Documents\GitHub\dataPreprocessing\source code\..\csv\step4_patient_aggregated.xlsx
✅ Done! Dataset is ready.
